In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

from fraud_engine.data.load import DEFAULT_CONFIG_PATH, load_config

In [ ]:
# A notebook's cwd is unreliable - anchor to the repo root instead.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

config = load_config(ROOT / DEFAULT_CONFIG_PATH)

# config.yaml paths are repo-root-relative.
interim_path = ROOT / config["paths"]["interim"]

In [ ]:
# Hold back the tail: Phase 02 carves the test set from it, and a boundary chosen
# after seeing its fraud rate is contaminated. Phase 02 sets the real number.
EDA_MAX_DAY = 120
EDA_FILTER = [("day", "<=", EDA_MAX_DAY)]


# One place enforces the horizon - a bare read_parquet later would span all 182 days.
def read_eda(columns):
    return pd.read_parquet(interim_path, columns=columns, filters=EDA_FILTER)

### Section 1 - base rate and time span

How much fraud there is, and over how long — measured inside the EDA horizon, not
across the full file. The base rate fixes the class imbalance every later design
decision has to survive, and is the reason accuracy is never reported here.

In [ ]:
df = read_eda(["day", "isFraud", "TransactionAmt"])

row_count = len(df)
print(f"Row count: {row_count}")

day_span = {
    "first_day": int(df["day"].min()),
    "last_day": int(df["day"].max()),
    "distinct_day_count": df["day"].nunique(),
}
print(f"Day span: {day_span}")

fraud_count = (df["isFraud"] == 1).sum()
print(f"Fraud count: {fraud_count}")

fraud_rate = fraud_count / row_count
print(f"Fraud rate: {fraud_rate:.3%}")

#### What the numbers say

414,542 transactions over 120 days (days 1–120 of the file's 182), no missing days.
14,600 of them are fraud — a base rate of **3.522%**.

Two consequences follow directly from that rate.

**Accuracy is unusable.** A model that predicts "not fraud" for every row scores
**96.478%**. Any accuracy figure quoted for this problem is describing the class balance,
not the model, which is why PR-AUC and recall@capacity are the metrics here.

**Manual review cannot be the answer on its own.** At ~3,455 transactions/day and the
`review_capacity: 0.01` committed in `cost_matrix.yaml`, the queue holds ~35 reviews/day
against ~122 frauds/day. Even a *perfect* ranker that spent every review slot on a true
fraud would top out at **28.4% recall**. The remaining ~72% has to be handled by the
allow/block decision, which is the constraint the Phase 06 cost policy exists to resolve.

`day` is an offset from an unpublished reference point, so this is a duration, not a date
range — no calendar claims can be made from it.

### Section 2 - fraud rate over time

Daily fraud rate across the horizon, with a 7-day centred rolling mean over the raw
series and transaction volume on the panel beneath — a rate move that coincides with
a volume move has a different explanation from one that does not.

The figure is saved to `reports/figures/`. Whether the
rate holds steady or drifts is the evidence the Phase 02 temporal split rests on.

In [ ]:
daily = df.groupby("day").agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
print(daily.shape)

In [ ]:
# The rate is only interpretable next to the volume that produced it.
fig, (ax_rate, ax_vol) = plt.subplots(2, 1, sharex=True, figsize=(11, 6), height_ratios=[2, 1])

# At ~3,000 transactions/day the raw series is mostly binomial noise.
ax_rate.plot(daily.index, daily["rate"], lw=1, alpha=0.35, label="daily")
ax_rate.plot(
    daily.index,
    daily["rate"].rolling(7, center=True).mean(),
    lw=2,
    label="7-day rolling mean (centred)",
)

# From zero: autoscale would amplify the noise into a trend.
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
ax_rate.set_title(f"Fraud rate over time (days 1-{EDA_MAX_DAY})")
ax_rate.legend(loc="upper left", frameon=False)
ax_rate.grid(alpha=0.25)

ax_vol.fill_between(daily.index, daily["volume"], alpha=0.4, lw=0)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
# `day` is an offset from an unpublished reference, not a calendar date.
ax_vol.set_xlabel("day (relative to an unpublished reference, not a calendar date)")
ax_vol.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

#### What the chart shows

The fraud rate is **not stationary** across the window.

Days 1–22 show volume climbing 55% while the daily fraud *count* stays flat (−3%), so the
rate falls to a 1.83% trough purely by dilution — the surge is legitimate customers, not
quieter attackers. Volume peaks and the rate bottoms on the same day (22), giving a
smoothed correlation of −0.88 over this stretch.

After that the series steps to a new level rather than continuing to trend. The 30-day
means for days 31–60, 61–90 and 91–120 are 4.03%, 4.01% and 4.02% — but those flat
averages hide real movement inside them. Days 91–120 rise from 3.41% to 4.79%, driven by a
20% increase in daily fraud count against a 25% fall in legitimate volume. That rise is
*not* dilution: it is more fraud against a smaller base.

The 30-day window width is a choice, and the flatness of those means is partly an artifact
of it.

**Implication for Phase 02.** A random split would scatter both regimes across train and
test, letting the model learn from a fraud environment it would not have had in
production. The split must be temporal.

### Section 3 - hour alignment and time of day

`TransactionDT` counts seconds from a reference point Vesta never published, so `hour`
(`seconds // 3600 % 24`) is a consistent 24-hour cycle but not necessarily wall-clock time.
Bucket 0 is midnight only if that reference is midnight-aligned — `min(TransactionDT)` is
exactly 86,400, one whole day, which hints at it but is an inference, not a fact.

**The volume curve is the test.** Human commerce has a deep overnight trough. If one appears
at plausible night hours, `hour` means hour-of-day and time-of-day claims are legitimate. If
the curve is flat or oddly phased, `hour` stays a usable cyclic feature but no time-of-day
claim can be made from it — and the verdict goes back into the `add_time_columns` docstring.

In [ ]:
hourly = (
    read_eda(["hour", "isFraud"])
    .groupby("hour")
    .agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
)
print(hourly.shape)

In [ ]:
# Volume on top: this curve is the alignment test, the rate is only interpretable once it
# is settled. No smoothing - each bucket holds ~19,000 transactions (noise is +/-0.15pp),
# and rolling() would treat hours 0 and 23 as distant rather than adjacent, blanking
# exactly the midnight window being tested.
fig, (ax_vol, ax_rate) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))

ax_vol.bar(hourly.index, hourly["volume"], width=0.85, alpha=0.7)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
ax_vol.set_title(f"Volume and fraud rate by hour bucket (days 1-{EDA_MAX_DAY})")
ax_vol.grid(alpha=0.25)

ax_rate.plot(hourly.index, hourly["rate"], lw=2, marker="o", ms=4)
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
# Bucket 0 is midnight only if the unpublished reference is midnight-aligned.
ax_rate.set_xlabel("hour bucket (0-23; bucket 0 is not necessarily midnight)")
ax_rate.set_xticks(range(0, 24, 2))
ax_rate.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

#### What the chart shows

**Bucket 0 is not midnight.** Volume swings 19× between bucket 9 (1,580) and bucket 19
(30,242) — an unmistakable diurnal cycle, but phased wrong for bucket 0 to be midnight,
which would mean the fewest transactions at 9am and the most between 7pm and 1am.

Placing the trough at a plausible pre-dawn hour puts midnight at **bucket 4–6**. Past
bucket 6 the mapping breaks: at bucket 8 the daily low lands at 1am, at bucket 12 at 9pm.
An offset of roughly +5 hours matches US Eastern's offset from UTC, which would be the
expected artifact of UTC timestamps against a mostly-US customer base — a hypothesis with
a mechanism, not a documented fact.

Consequence: `hour` is a **cyclic feature, not a wall-clock label**, and no time-of-day
claim can be made without carrying that offset as a stated assumption. The verdict is
recorded in the `add_time_columns` docstring, which previously left the question open.

#### The rate peak is a denominator effect

The fraud rate peaks at bucket 8 (10.01%, against 3.45% across buckets 17–23), and that
peak sits exactly where volume is lowest. But the counts run the other way:

| | frauds/day | legit/day | rate |
|---|---|---|---|
| bucket 8 | 1.6 | 15 | 10.01% |
| buckets 17–23 | 8.5 | 237 | 3.45% |

Relative to the plateau, fraud falls to 0.19× while legitimate activity falls to 0.06×.
Both collapse overnight; legitimate activity collapses harder. **Fraud does not peak there
— it declines more slowly than everything around it.** The hours with the most fraud by
count are the plateau hours, which carry roughly five times more fraud per day.

This is section 2's mechanism inverted: there, a surge in legitimate volume diluted a flat
fraud count; here, a collapse in legitimate volume concentrates a falling one. Rate and
count answer different questions, and only the count says where the money is.